# Pre-compute Mel Spectrograms
One-time job: convert all audio (clean clips + soundscape segments) into `.npy` files.
This eliminates on-the-fly audio decoding and FFT computation during training.
Using `.npy` over `.pt` for faster loading (raw binary read vs pickle deserialization).

In [1]:
import os
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

CFG = {
    'base_dir': '/data/scratch/scienceteam/jupyter-mir/bird/Data/birdclef-2026',
    'pseudo_path': '/data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed/pseudo_labels.csv',
    'spec_dir': '/data/scratch/scienceteam/jupyter-mir/bird/Data/precomputed_specs',
    'sr': 32000,
    'duration': 20,
    'n_fft': 4096,
    'hop_length': 1252,
    'n_mels': 224,
    'fmin': 0,
    'fmax': 16000,
    'top_db': 80.0,
    'img_width': 512,
    'num_workers': 48,
}

TARGET_LEN = CFG['sr'] * CFG['duration']  # 640000 samples

CLEAN_SPEC_DIR = os.path.join(CFG['spec_dir'], 'train_audio')
SC_SPEC_DIR = os.path.join(CFG['spec_dir'], 'soundscapes')
os.makedirs(CLEAN_SPEC_DIR, exist_ok=True)
os.makedirs(SC_SPEC_DIR, exist_ok=True)

print(f"Output dirs:")
print(f"  Clean: {CLEAN_SPEC_DIR}")
print(f"  Soundscapes: {SC_SPEC_DIR}")

Output dirs:
  Clean: /data/scratch/scienceteam/jupyter-mir/bird/Data/precomputed_specs/train_audio
  Soundscapes: /data/scratch/scienceteam/jupyter-mir/bird/Data/precomputed_specs/soundscapes


In [2]:
def audio_to_melspec(y, cfg):
    """Convert waveform to normalized 3-channel mel spectrogram tensor."""
    mel = librosa.feature.melspectrogram(
        y=y, sr=cfg['sr'],
        n_fft=cfg['n_fft'], hop_length=cfg['hop_length'],
        n_mels=cfg['n_mels'], fmin=cfg['fmin'], fmax=cfg['fmax'],
    )
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=cfg['top_db'])
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    if mel_db.shape[1] < cfg['img_width']:
        mel_db = np.pad(mel_db, ((0, 0), (0, cfg['img_width'] - mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :cfg['img_width']]

    mel_3ch = np.stack([mel_db, mel_db, mel_db], axis=0)
    return mel_3ch.astype(np.float16)


def process_clean_clip(args):
    """Process a single clean training clip: load full audio, take center/random 20s, save spectrogram."""
    filepath, filename, out_dir, cfg, target_len = args
    stem = Path(filename).stem
    # Preserve subfolder structure (species/filename)
    species_dir = os.path.dirname(filename)
    save_dir = os.path.join(out_dir, species_dir)
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{stem}.npy")

    if os.path.exists(save_path):
        return 'skip'

    try:
        y, _ = librosa.load(filepath, sr=cfg['sr'])
    except Exception:
        y = np.zeros(target_len, dtype=np.float32)

    if len(y) == 0:
        y = np.zeros(target_len, dtype=np.float32)
    if len(y) < target_len:
        reps = (target_len // len(y)) + 1
        y = np.tile(y, reps)

    # Take center crop (deterministic for pre-computation)
    center = len(y) // 2
    half = target_len // 2
    y = y[center - half : center - half + target_len]

    max_val = np.abs(y).max()
    if max_val > 0:
        y = y / max_val

    spec = audio_to_melspec(y.astype(np.float32), cfg)
    np.save(save_path, spec)
    return 'ok'


def process_soundscape_segment(args):
    """Process a single soundscape segment: load audio, extract 20s centered on 5s segment, save spectrogram."""
    filepath, soundscape_id, start_sec, out_dir, cfg, target_len = args
    save_path = os.path.join(out_dir, f"{soundscape_id}_{int(start_sec)}.npy")

    if os.path.exists(save_path):
        return 'skip'

    try:
        y, _ = librosa.load(filepath, sr=cfg['sr'])
    except Exception:
        y = np.zeros(cfg['sr'] * 60, dtype=np.float32)

    start_sample = int(start_sec) * cfg['sr']
    center_sample = start_sample + (cfg['sr'] * 5 // 2)
    half_window = target_len // 2
    chunk_start = max(0, center_sample - half_window)
    chunk_end = chunk_start + target_len

    if chunk_end > len(y):
        chunk_start = max(0, len(y) - target_len)
        chunk_end = chunk_start + target_len

    segment = y[chunk_start:chunk_end]
    if len(segment) < target_len:
        segment = np.pad(segment, (0, target_len - len(segment)))

    max_val = np.abs(segment).max()
    if max_val > 0:
        segment = segment / max_val

    spec = audio_to_melspec(segment.astype(np.float32), cfg)
    np.save(save_path, spec)
    return 'ok'


print("Helper functions defined.")

Helper functions defined.


In [6]:
# Pre-compute clean clips (35,549 files)

train = pd.read_csv(os.path.join(CFG['base_dir'], 'train.csv'))
train['filepath'] = train['filename'].apply(
    lambda x: os.path.join(CFG['base_dir'], 'train_audio', x)
)

print(f"Processing {len(train):,} clean clips with {CFG['num_workers']} workers...")

tasks = [
    (row['filepath'], row['filename'], CLEAN_SPEC_DIR, CFG, TARGET_LEN)
    for _, row in train.iterrows()
]

ok_count = 0
skip_count = 0
err_count = 0

with ProcessPoolExecutor(max_workers=CFG['num_workers']) as executor:
    futures = {executor.submit(process_clean_clip, t): t for t in tasks}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Clean clips"):
        try:
            result = future.result()
            if result == 'ok':
                ok_count += 1
            else:
                skip_count += 1
        except Exception as e:
            err_count += 1

print(f"\nClean clips done: {ok_count:,} created, {skip_count:,} skipped (already exist), {err_count:,} errors")

Processing 35,549 clean clips with 48 workers...


Clean clips: 100%|██████████| 35549/35549 [10:30<00:00, 56.35it/s]



Clean clips done: 35,549 created, 0 skipped (already exist), 0 errors


In [4]:
# Pre-compute soundscape segments (pseudo-labeled + labeled)

pseudo_df = pd.read_csv(CFG['pseudo_path'])
pseudo_df['filepath'] = pseudo_df['soundscape_id'].apply(
    lambda x: os.path.join(CFG['base_dir'], 'train_soundscapes', x + '.ogg')
)

sc_labels = pd.read_csv(os.path.join(CFG['base_dir'], 'train_soundscapes_labels.csv'))
sc_labels['soundscape_id'] = sc_labels['filename'].str.replace('.ogg', '', regex=False)
sc_labels['filepath'] = sc_labels['filename'].apply(
    lambda x: os.path.join(CFG['base_dir'], 'train_soundscapes', x)
)

def parse_time(t):
    parts = t.split(':')
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

sc_labels['start_sec'] = sc_labels['start'].apply(parse_time)

# Combine both into one task list (some segments overlap -- same file, different start)
all_segments = pd.concat([
    pseudo_df[['filepath', 'soundscape_id', 'start_sec']],
    sc_labels[['filepath', 'soundscape_id', 'start_sec']],
], ignore_index=True)

# Deduplicate: same soundscape_id + start_sec only needs one spectrogram
all_segments = all_segments.drop_duplicates(subset=['soundscape_id', 'start_sec'])

print(f"Processing {len(all_segments):,} unique soundscape segments with {CFG['num_workers']} workers...")

tasks = [
    (row['filepath'], row['soundscape_id'], row['start_sec'], SC_SPEC_DIR, CFG, TARGET_LEN)
    for _, row in all_segments.iterrows()
]

ok_count = 0
skip_count = 0
err_count = 0

with ProcessPoolExecutor(max_workers=CFG['num_workers']) as executor:
    futures = {executor.submit(process_soundscape_segment, t): t for t in tasks}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Soundscape segments"):
        try:
            result = future.result()
            if result == 'ok':
                ok_count += 1
            else:
                skip_count += 1
        except Exception as e:
            err_count += 1

print(f"\nSoundscape segments done: {ok_count:,} created, {skip_count:,} skipped, {err_count:,} errors")

Processing 127,843 unique soundscape segments with 48 workers...


Soundscape segments: 100%|██████████| 127843/127843 [40:38<00:00, 52.42it/s] 



Soundscape segments done: 127,843 created, 0 skipped, 0 errors


In [5]:
# Verify: count files and spot-check shapes

import glob

clean_files = glob.glob(os.path.join(CLEAN_SPEC_DIR, '**', '*.npy'), recursive=True)
sc_files = glob.glob(os.path.join(SC_SPEC_DIR, '*.npy'))

print(f"Clean spec files: {len(clean_files):,}  (expected: {len(train):,})")
print(f"Soundscape spec files: {len(sc_files):,}  (expected: {len(all_segments):,})")

# Spot check
for f in clean_files[:3]:
    arr = np.load(f)
    print(f"  {os.path.basename(f)}: shape={arr.shape}, dtype={arr.dtype}, min={arr.min():.4f}, max={arr.max():.4f}")

for f in sc_files[:3]:
    arr = np.load(f)
    print(f"  {os.path.basename(f)}: shape={arr.shape}, dtype={arr.dtype}, min={arr.min():.4f}, max={arr.max():.4f}")

# Disk usage estimate
sample_size = os.path.getsize(clean_files[0]) / (1024**2)
total_files = len(clean_files) + len(sc_files)
print(f"\nSample file size: {sample_size:.2f} MB")
print(f"Estimated total disk usage: {sample_size * total_files / 1024:.1f} GB")
print("\nDone! Ready for C2 v2 training.")

Clean spec files: 35,549  (expected: 35,549)
Soundscape spec files: 127,843  (expected: 127,843)
  iNat803788.npy: shape=(3, 224, 512), dtype=float16, min=0.0000, max=1.0000
  iNat341969.npy: shape=(3, 224, 512), dtype=float16, min=0.0000, max=1.0000
  XC619234.npy: shape=(3, 224, 512), dtype=float16, min=0.0000, max=1.0000
  BC2026_Train_8725_S22_20230101_020000_50.npy: shape=(3, 224, 512), dtype=float16, min=0.0000, max=1.0000
  BC2026_Train_5809_S13_20221028_020000_50.npy: shape=(3, 224, 512), dtype=float16, min=0.0000, max=1.0000
  BC2026_Train_3932_S02_20221224_181500_20.npy: shape=(3, 224, 512), dtype=float16, min=0.0000, max=1.0000

Sample file size: 0.66 MB
Estimated total disk usage: 104.7 GB

Done! Ready for C2 v2 training.
